In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    T5ForConditionalGeneration, BartForConditionalGeneration,
    Trainer, TrainingArguments, DataCollatorForSeq2Seq
)
from datasets import Dataset as HFDataset
import numpy as np
from sklearn.metrics import accuracy_score
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import Levenshtein
import json
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")



class VietnameseTextCorrectionTrainer:
    """Main trainer class"""

    def __init__(self, model_name, output_dir="./results"):
        self.model_name = model_name
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(exist_ok=True)

        # Load tokenizer và model
        print(f"Loading {model_name}...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

        # Thêm special tokens nếu cần
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token



    def prepare_data(self,data_path):
        """Chuẩn bị dữ liệu train/val/test"""
        print("Preparing dataset...")

        # Load toàn bộ data
        all_data = []
        with open(data_path, 'r', encoding='utf-8') as f:
            for line in f:
                if '\t' in line:
                    corrupted, original = line.strip().split('\t', 1)
                    all_data.append({
                        'input_text': f"sửa lỗi: {corrupted}",
                        'target_text': original
                    })

        np.random.shuffle(all_data)
        data = HFDataset.from_list(all_data)
        data = data.map(self._tokenize_function, batched=True)
        print(f"Dataset : {len(data)}")

        return data

    def _tokenize_function(self, examples):
        """Tokenize function cho HuggingFace Dataset"""
        inputs = self.tokenizer(
            examples['input_text'],
            max_length=512,
            truncation=True,
            padding=False
        )

        targets = self.tokenizer(
            examples['target_text'],
            max_length=512,
            truncation=True,
            padding=False
        )

        inputs['labels'] = targets['input_ids']
        return inputs

    def train(self,train_dataset,val_dataset, num_epochs=3, batch_size=16, learning_rate=5e-5):
        """Training mô hình"""
        print(f"Starting training for {num_epochs} epochs...")


        training_args = TrainingArguments(
            output_dir=str(self.output_dir),
            num_train_epochs=num_epochs,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            warmup_steps=50,
            weight_decay=0.01,
            logging_dir=str(self.output_dir / 'logs'),
            logging_steps=50,
            eval_steps=500,
            save_steps=500,
            eval_strategy="steps",
            save_strategy="steps",
            load_best_model_at_end=True,
            metric_for_best_model="eval_loss",
            greater_is_better=False,
            fp16=True,
            learning_rate=learning_rate,
            gradient_accumulation_steps=2,
        )

        data_collator = DataCollatorForSeq2Seq(
            tokenizer=self.tokenizer,
            model=self.model,
            padding=True
        )

        trainer = Trainer(
            model=self.model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            tokenizer=self.tokenizer,
            data_collator=data_collator,
        )

        # Train
        trainer.train()
        # Save model
        trainer.save_model(str(self.output_dir / 'final_model'))
        self.tokenizer.save_pretrained(str(self.output_dir / 'final_model'))
        print(f"Training completed! Model saved to {self.output_dir / 'final_model'}")
        return trainer

In [ ]:
trainer = VietnameseTextCorrectionTrainer(
    model_name="VietAI/vit5-base",
    output_dir=f"./results_vit5_base_V1"
)
# 3. Prepare data
train_dataset = trainer.prepare_data("/content/train_data.txt")

val_dataset = trainer.prepare_data("/content/val_data.txt")

# 4. Train model
trainer.train(train_dataset,val_dataset,num_epochs=3, batch_size=16, learning_rate=3e-5)

In [ ]:
!huggingface-cli login
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# 1. Đường dẫn tới thư mục đã lưu model
local_dir = "/content/results_vit5_base_V1/final_model"

# 2. Load lại tokenizer + model từ local
tokenizer = AutoTokenizer.from_pretrained(local_dir)
model     = AutoModelForSeq2SeqLM.from_pretrained(local_dir)


# 3. Đẩy lên Hugging Face Hub
repo_id = "TranKhang/DACNTT_viT5_base_V1"


model.push_to_hub(repo_id)
tokenizer.push_to_hub(repo_id)


In [ ]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import json
from tqdm import tqdm

def evaluate_model_with_logprob(
    data_path: str,
    save_path: str = "results.json",
    batch_size: int = 16,
    num_beams: int = 6,
    num_return_sequences: int = 1  # Có thể chỉnh thành 3 nếu muốn xem top-k
):
    # Load model và tokenizer
    model_name = "TranKhang/DACNTT_viT5_base_V1"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))
    model.eval()

    # Đọc dữ liệu
    with open(data_path, "r", encoding="utf-8") as f:
        samples = [line.strip().split("\t") for line in f if "\t" in line]
    corrupted_texts, original_texts = zip(*samples)

    print(f"📄 Total samples: {len(corrupted_texts)}")
    print("🚀 Evaluating model...")

    results = []
    for i in tqdm(range(0, len(corrupted_texts), batch_size), desc="Processing batches"):
        batch_corrupted = corrupted_texts[i:i+batch_size]
        batch_original = original_texts[i:i+batch_size]

        # Chuẩn bị input
        inputs = tokenizer(
            [f"sửa lỗi: {text}" for text in batch_corrupted],
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to(model.device)

        # Generate với log-prob
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_length=512,
                num_beams=num_beams,
                num_return_sequences=num_return_sequences,
                length_penalty=0.8,
                early_stopping=True,
                output_scores=True,
                return_dict_in_generate=True
            )

        # Decode kết quả
        predictions = tokenizer.batch_decode(outputs.sequences, skip_special_tokens=True)

        for j in range(len(batch_corrupted)):
            if num_return_sequences > 1:
                start_idx = j * num_return_sequences
                end_idx = start_idx + num_return_sequences
                batch_preds = predictions[start_idx:end_idx]
                batch_scores = outputs.sequences_scores[start_idx:end_idx].tolist()

                best_idx = torch.argmax(outputs.sequences_scores[start_idx:end_idx]).item()
                best_pred = batch_preds[best_idx]
                best_score = batch_scores[best_idx]
            else:
                best_pred = predictions[j]
                best_score = outputs.sequences_scores[j].item()

            results.append({
                "Corrupted text": batch_corrupted[j],
                "Prediction": best_pred,
                "Original text": batch_original[j],
            })

    # Lưu kết quả
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    print(f"✅ Results saved to {save_path}")

# Chạy đánh giá
evaluate_model_with_logprob(
    data_path="test_data.txt",
    save_path="evaluation_results.json",
    batch_size=8,
    num_beams=6,
    num_return_sequences=1
)

In [15]:
import json
import re
from difflib import SequenceMatcher
from typing import List, Dict, Tuple, Any
import string

def normalize_text(text: str) -> str:
    """Normalize text for comparison"""
    text = re.sub(r'\s+', ' ', text.strip())
    return text

def get_edit_operations(text1: str, text2: str) -> List[Tuple]:
    """Get edit operations between two texts"""
    matcher = SequenceMatcher(None, text1, text2)
    operations = []

    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag != 'equal':
            operations.append({
                'type': tag,
                'text1_start': i1,
                'text1_end': i2,
                'text2_start': j1,
                'text2_end': j2,
                'text1_content': text1[i1:i2],
                'text2_content': text2[j1:j2]
            })

    return operations

def evaluate_text_correction(original: str, predicted: str, corrupted: str) -> Dict:
    """
    Evaluate text correction performance using edit operations
    """

    print(f"Original:  '{original}'")
    print(f"Corrupted: '{corrupted}'")
    print(f"Predicted: '{predicted}'")
    print()

    actual_errors = get_edit_operations(corrupted, original)
    print(f"Actual errors: {len(actual_errors)}")
    for i, error in enumerate(actual_errors):
        print(f"  {i+1}. {error['type']}: '{error['text1_content']}' -> '{error['text2_content']}'")
    print()

    model_changes = get_edit_operations(corrupted, predicted)
    print(f"Model changes: {len(model_changes)}")
    for i, change in enumerate(model_changes):
        print(f"  {i+1}. {change['type']}: '{change['text1_content']}' -> '{change['text2_content']}'")
    print()

    if predicted == original:
        print("Perfect correction detected!")
        return {
            'detection_precision': 1.0,
            'detection_recall': 1.0,
            'detection_f1': 1.0,
            'correction_precision': 1.0,
            'correction_recall': 1.0,
            'correction_f1': 1.0,
            'actual_errors': len(actual_errors),
            'detected_errors': len(model_changes),
            'true_detections': len(actual_errors),
            'true_corrections': len(actual_errors),
            'perfect_correction': True
        }

    true_detections = 0
    true_corrections = 0


    if len(model_changes) > 0:
        original_distance = sum(1 for op in actual_errors)
        predicted_distance = len(get_edit_operations(predicted, original))

        if predicted_distance < original_distance:
            true_detections = min(len(model_changes), len(actual_errors))
            true_corrections = len(actual_errors) - predicted_distance

    detection_precision = true_detections / len(model_changes) if len(model_changes) > 0 else 0
    detection_recall = true_detections / len(actual_errors) if len(actual_errors) > 0 else 0
    detection_f1 = (2 * detection_precision * detection_recall /
                   (detection_precision + detection_recall)) if (detection_precision + detection_recall) > 0 else 0

    correction_precision = true_corrections / len(model_changes) if len(model_changes) > 0 else 0
    correction_recall = true_corrections / len(actual_errors) if len(actual_errors) > 0 else 0
    correction_f1 = (2 * correction_precision * correction_recall /
                    (correction_precision + correction_recall)) if (correction_precision + correction_recall) > 0 else 0

    return {
        'detection_precision': detection_precision,
        'detection_recall': detection_recall,
        'detection_f1': detection_f1,
        'correction_precision': correction_precision,
        'correction_recall': correction_recall,
        'correction_f1': correction_f1,
        'actual_errors': len(actual_errors),
        'detected_errors': len(model_changes),
        'true_detections': true_detections,
        'true_corrections': true_corrections,
        'perfect_correction': False
    }

def evaluate_json_file(json_data: List[Dict]) -> Dict:
    """
    Evaluate multiple samples from JSON data
    """
    all_results = []
    total_metrics = {
        'detection_precision': 0,
        'detection_recall': 0,
        'detection_f1': 0,
        'correction_precision': 0,
        'correction_recall': 0,
        'correction_f1': 0,
        'perfect_corrections': 0
    }

    print("=" * 80)
    print("EVALUATING ALL SAMPLES")
    print("=" * 80)

    for i, sample in enumerate(json_data):
        print(f"\n{'='*50}")
        print(f"SAMPLE {i+1}")
        print(f"{'='*50}")

        original = sample["Original text"]
        predicted = sample["Prediction"]
        corrupted = sample["Corrupted text"]

        results = evaluate_text_correction(original, predicted, corrupted)
        all_results.append(results)

        for key in total_metrics:
            if key == 'perfect_corrections':
                total_metrics[key] += 1 if results.get('perfect_correction', False) else 0
            else:
                total_metrics[key] += results[key]

    num_samples = len(json_data)
    avg_metrics = {}
    for key, total in total_metrics.items():
        if key == 'perfect_corrections':
            avg_metrics[key] = total
            avg_metrics['perfect_correction_rate'] = total / num_samples
        else:
            avg_metrics[key] = total / num_samples

    print(f"\n{'='*80}")
    print("OVERALL RESULTS")
    print(f"{'='*80}")
    print(f"Total samples: {num_samples}")
    print()
    print("Average Metrics:")
    for key, value in avg_metrics.items():
        if key not in ['perfect_corrections', 'perfect_correction_rate']:
            print(f"  {key}: {value:.4f}")

    return {
        'individual_results': all_results,
        'average_metrics': avg_metrics,
        'summary': {
            'total_samples': num_samples,
            'perfect_corrections': avg_metrics['perfect_corrections'],
            'perfect_correction_rate': avg_metrics['perfect_correction_rate']
        }
    }



def load_and_evaluate_json_file(file_path: str):
    """Load and evaluate JSON file"""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            json_data = json.load(f)
        return evaluate_json_file(json_data)
    except FileNotFoundError:
        print(f"File {file_path} not found!")
        return None
    except json.JSONDecodeError:
        print(f"Invalid JSON format in {file_path}!")
        return None

results = load_and_evaluate_json_file("/content/evaluation_results.json")

Kết quả truyền trực tuyến bị cắt bớt đến 5000 dòng cuối.
  2. replace: 'u' -> 'ư'

Model changes: 2
  1. replace: 't' -> 'T'
  2. replace: 'u' -> 'ư'

Perfect correction detected!

SAMPLE 4725
Original:  '3. Trách nhiệm của tổ chức tín dụng phi ngân hàng về tài sản, quyền, nghĩa vụ và các lợi ích liên quan của chi nhánh, văn phòng đại diện, đơn vị sự nghiệp chấm dứt hoạt động, giải thể.”'
Corrupted: '3. Trách nhiệm của tô chức tín dụng phi ngân hàng về tài sản, quyền, nghĩa vụ và các lợi ích liên quan của chi nhánh, văn phòng đại diện, đơn vị sự nghiệp chấm dứt hoạt động, giải thể.”'
Predicted: '3. Trách nhiệm của tổ chức tín dụng phi ngân hàng về tài sản, quyền, nghĩa vụ và các lợi ích liên quan của chi nhánh, văn phòng đại diện, đơn vị sự nghiệp chấm dứt hoạt động, giải thể.'

Actual errors: 1
  1. replace: 'ô' -> 'ổ'

Model changes: 2
  1. replace: 'ô' -> 'ổ'
  2. delete: '”' -> ''


SAMPLE 4726
Original:  'Tự nguyện chấm dứt hoạt động, giải thể văn phòng đại diện, đơn vị sự nghiệp: